In [ ]:

!pip install pypdf sentence-transformers faiss-cpu numpy transformers


In [ ]:
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 60.1 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import pipeline
from typing import List
from textwrap import wrap as word_wrap

In [ ]:
def load_pdf(path: str) -> str:
    reader = PdfReader(path)
    full_text = ""
    for page in reader.pages:
        full_text += page.extract_text() or ""
    return full_text


def chunk_text(text: str,
               chunk_size: int = 500,
               overlap: int = 50) -> List[str]:

    words  = text.split()
    chunks = []
    start  = 0

    while start < len(words):
        end   = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap

    return chunks

In [ ]:
def build_embeddings(chunks: List[str],
                     model_name: str = "all-MiniLM-L6-v2"):

    print(f"  Loading embedding model: {model_name} ...")
    embedder = SentenceTransformer(model_name)

    print(f"  Embedding {len(chunks)} chunks ...")
    embeddings = embedder.encode(chunks, show_progress_bar=True)

    return embedder, embeddings.astype("float32")

In [ ]:
def build_faiss_index(embeddings: np.ndarray) -> faiss.Index:
    dimension = embeddings.shape[1]

    index = faiss.IndexFlatL2(dimension)

    index.add(embeddings)
    print(f"  FAISS index built: {index.ntotal} vectors stored.")

    return index

In [ ]:
def retrieve(query: str,
             embedder,
             index: faiss.Index,
             chunks: List[str],
             top_k: int = 3) -> List[str]:

    query_vec = embedder.encode([query]).astype("float32")

    distances, indices = index.search(query_vec, top_k)

    retrieved = [chunks[idx] for idx in indices[0] if idx != -1]
    return retrieved

In [ ]:
def generate_answer(query: str,
                    context_chunks: List[str],
                    generator,
                    max_tokens: int = 300) -> str:

    context = "\n\n---\n\n".join(context_chunks)

    prompt = f"""Answer the question using only the context below.
If the answer is not in the context, say
"I don't know based on the provided documents."

Context:
{context}

Question: {query}

Answer:"""

    result = generator(prompt, max_new_tokens=max_tokens)
    return result[0]["generated_text"].strip()

In [ ]:
def build_rag_pipeline(pdf_paths: List[str]):
    all_chunks = []
    for pdf_path in pdf_paths:
        print(f"\n[+] Loading: {pdf_path}")
        text   = load_pdf(pdf_path)
        chunks = chunk_text(text, chunk_size=500, overlap=50)
        print(f"    -> {len(chunks)} chunks extracted")
        all_chunks.extend(chunks)

    print(f"\n[+] Total chunks: {len(all_chunks)}")

    print("\n[+] Building embeddings ...")
    embedder, embeddings = build_embeddings(all_chunks)

    print("\n[+] Building FAISS index ...")
    index = build_faiss_index(embeddings)

    return embedder, index, all_chunks


def ask(query: str, embedder, index, chunks, generator, top_k: int = 3):
    print(f"\n{'='*60}")
    print(f"QUESTION: {query}")
    print(f"{'='*60}")

    print("\n[Retrieve] Searching FAISS index ...")
    context = retrieve(query, embedder, index, chunks, top_k=top_k)
    print(f"  -> {len(context)} chunks retrieved")

    print("\n[Generate] Calling flan-t5-base ...")
    answer = generate_answer(query, context, generator)

    print("\nANSWER:\n")
    for line in word_wrap(answer, width=70):
        print(" ", line)
    return answer

In [ ]:
PDF_FILES = [
    "5b_pdf.pdf",
    "5b_pdf2.pdf",
]

embedder, index, chunks = build_rag_pipeline(PDF_FILES)

print("\n[+] Loading local LLM: flan-t5-base ...")
generator = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

questions = [
    "What is the core idea behind the attention mechanism?",
    "How does BERT differ from traditional language models?",
    "What datasets were used for pre-training?",
]

for q in questions:
    ask(q, embedder, index, chunks, generator)


[+] Loading: 5b_pdf.pdf
    -> 14 chunks extracted

[+] Loading: 5b_pdf2.pdf
    -> 23 chunks extracted

[+] Total chunks: 37

[+] Building embeddings ...
  Loading embedding model: all-MiniLM-L6-v2 ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Embedding 37 chunks ...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]


[+] Building FAISS index ...
  FAISS index built: 37 vectors stored.

[+] Loading local LLM: flan-t5-base ...


...


ANSWER:

  Answer the question using only the context below. If the answer is not
  in the context, say "I don't know based on the provided documents."
  Context: and W O ∈ Rhdv×dmodel. In this work we employ h = 8 parallel
  attention layers, or heads. For each of these we use dk = dv =
  dmodel/h = 64. Due to the reduced dimension of each head, the total
  computational cost is similar to that of single-head attention with
  full dimensionality. 3.2.3 Applications of Attention in our Model The
  Transformer uses multi-head attention in three different ways: • In
  "encoder-decoder attention" layers, the queries come from the previous
  decoder layer, and the memory keys and values come from the output of
  the encoder. This allows every position in the decoder to attend over
  all positions in the input sequence. This mimics the typical encoder-
  decoder attention mechanisms in sequence-to-sequence models such as
  [38, 2, 9]. • The encoder contains self-attention layers. In a self

[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ANSWER:

  Answer the question using only the context below. If the answer is not
  in the context, say "I don't know based on the provided documents."
  Context: 4We note that in the literature the bidirectional Trans-
  Input/Output Representations To make BERT handle a variety of down-
  stream tasks, our input representation is able to unambiguously
  represent both a single sentence and a pair of sentences (e.g.,⟨
  Question, Answer⟩) in one token sequence. Throughout this work, a
  “sentence” can be an arbi- trary span of contiguous text, rather than
  an actual linguistic sentence. A “sequence” refers to the in- put
  token sequence to BERT, which may be a sin- gle sentence or two
  sentences packed together. We use WordPiece embeddings (Wu et al.,
  2016) with a 30,000 token vocabulary. The ﬁrst token of every sequence
  is always a special clas- siﬁcation token ( [CLS]). The ﬁnal hidden
  state corresponding to this token is used as the ag- gregate sequence
  representation f

[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ANSWER:

  Answer the question using only the context below. If the answer is not
  in the context, say "I don't know based on the provided documents."
  Context: down-stream tasks. During ﬁne-tuning, all parameters are ﬁne-
  tuned. [CLS] is a special symbol added in front of every input
  example, and [SEP] is a special separator token (e.g. separating ques-
  tions/answers). ing and auto-encoder objectives have been used for
  pre-training such models (Howard and Ruder, 2018; Radford et al.,
  2018; Dai and Le, 2015). 2.3 Transfer Learning from Supervised Data
  There has also been work showing effective trans- fer from supervised
  tasks with large datasets, such as natural language inference (Conneau
  et al., 2017) and machine translation (McCann et al., 2017). Computer
  vision research has also demon- strated the importance of transfer
  learning from large pre-trained models, where an effective recipe is
  to ﬁne-tune models pre-trained with Ima- geNet (Deng et al., 2009;
  Y